# Building 10 Routing Integration Demo

This notebook is intentionally thin. Routing and integration logic now live in `src/navigation/*`.

In [ ]:
from pathlib import Path
import json
import pandas as pd

from src.navigation.localization_client import LocalizationClient
from src.navigation.pipeline import NavigationSession
from src.navigation.router_core import B10Router

In [ ]:
NAV_DIR = Path("indoor-navigation-part")
SCAN_CSV = Path("data/bldg10/final_data.csv")
LOCALIZATION_URL = "http://localhost:8080"
GOAL_ROOM_ID = "10.2.63"

In [ ]:
IMU_COLUMNS = [
    "accel_x", "accel_y", "accel_z",
    "gyro_x", "gyro_y", "gyro_z",
    "mag_x", "mag_y", "mag_z", "mag_heading",
]

df = pd.read_csv(SCAN_CSV)
row = df.iloc[0]

scan = {
    "rssi": {col: float(row[col]) for col in df.columns if col.startswith("AP")},
    "imu": {col: float(row[col]) if col in df.columns else 0.0 for col in IMU_COLUMNS},
    "top_k": 3,
}

In [ ]:
router = B10Router(NAV_DIR)
client = LocalizationClient(base_url=LOCALIZATION_URL)
session = NavigationSession(router=router, localization_client=client, confidence_threshold=0.60, top_k=3, transition_hits_required=2)

decision = session.update_and_route(scan=scan, goal_room_id=GOAL_ROOM_ID, accessible=True, visualize=False)
print(json.dumps(decision.as_dict(), indent=2))